# Industry-Specific LLM Bot

## Capstone Project

### Industry:
Education and Training

### Objective
The objective of this project is to build a simple industry-specific chatbot using a pre-trained Large Language Model (LLM) from Hugging Face. The chatbot is designed to answer questions related to Education and Training.

The project demonstrates how pre-trained language models can be used to create domain-specific conversational AI without building a model from scratch.

# Install required libraries

!pip -q install transformers accelerate torch gradio

# Import Libraries

In this section, we import all the required libraries for loading the language model and building the chatbot.

In [ ]:
import torch
from transformers import pipeline
import gradio as gr

# Dataset Preparation

A small education-specific dataset was created for fine-tuning the pre-trained FLAN-T5 model.

The dataset contains question-answer pairs related to programming, artificial intelligence, databases, and other educational topics.

This dataset enables the model to better understand educational queries and generate more relevant responses.

In [ ]:
from datasets import Dataset

data = {
    "question": [
        "What is Python?",
        "What is SQL?",
        "What is Machine Learning?",
        "What is Artificial Intelligence?",
        "What is Data Science?"
    ],

    "answer": [
        "Python is a high-level programming language.",
        "SQL is a language used to manage databases.",
        "Machine Learning is a subset of AI that allows computers to learn from data.",
        "Artificial Intelligence is the simulation of human intelligence by machines.",
        "Data Science is the process of extracting insights from data."
    ]
}

dataset = Dataset.from_dict(data)

dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 5
})

# Tokenization

The dataset is converted into tokens so that it can be processed by the FLAN-T5 model during fine-tuning.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def preprocess(example):

    text = "Question: " + example["question"]

    model_input = tokenizer(
        text,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        example["answer"],
        max_length=128,
        truncation=True
    )

    model_input["labels"] = labels["input_ids"]

    return model_input

tokenized_dataset = dataset.map(preprocess)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# Fine-Tuning the Model

The pre-trained FLAN-T5 model is fine-tuned using the education-specific dataset.

For demonstration purposes, the model is trained for one epoch, which is sufficient to illustrate the fine-tuning process.

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Step,Training Loss
1,2.416515
2,2.005759
3,2.179143


TrainOutput(global_step=3, training_loss=2.200472354888916, metrics={'train_runtime': 1.8617, 'train_samples_per_second': 2.686, 'train_steps_per_second': 1.611, 'total_flos': 57508918272.0, 'train_loss': 2.200472354888916, 'epoch': 1.0})

# Save the Fine-Tuned Model

After fine-tuning, the updated model is saved and later used by the chatbot for answering user queries.

In [ ]:
trainer.save_model("fine_tuned_flan_t5")

tokenizer.save_pretrained("fine_tuned_flan_t5")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fine_tuned_flan_t5/tokenizer_config.json',
 'fine_tuned_flan_t5/tokenizer.json')

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Load the fine-tuned model and tokenizer
model_path = "fine_tuned_flan_t5"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Define a function to generate text, mimicking the pipeline behavior
def generator(input_text, max_new_tokens=50, do_sample=True, temperature=0.7):
    inputs = tokenizer(input_text, return_tensors="pt")

    # Generate text using the model's generate method
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": generated_text}]


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


# Create Chatbot Function

The chatbot combines the user's question with an instruction prompt before sending it to the fine-tuned FLAN-T5 model.

The model then generates a concise answer related to Education and Technology.

In [ ]:
def chatbot(question):

    prompt = f"""
Answer the following question in a clear, detailed manner and elaborate each and everything.

Question: {question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return response[0]["generated_text"]

# Test the Chatbot

Let's test the chatbot using a sample question.

In [ ]:
chatbot("What is Machine Learning?")

'A machine learning algorithm is a mathematical algorithm based on the use of computers to learn facts.'

In [ ]:
chatbot("What is SQL?")

'SQL is a programming language that can be used to create and maintain databases.'

In [ ]:
chatbot("What is Artificial Intelligence?")

"Artificial intelligence is the development of algorithms to predict and process a user's actions."

# Build Gradio Interface

Gradio provides a simple web interface for interacting with the chatbot.

In [ ]:
demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="Education & Training LLM Bot",
    description="Ask any question related to Education and Training."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3bda3034b759d8e66.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Conclusion

In this project, a simple industry-specific chatbot was developed using a pre-trained Large Language Model from Hugging Face.

The chatbot is capable of answering questions related to Education and Training while restricting responses to its intended domain.

This project demonstrates the practical use of Large Language Models for building conversational AI applications with minimal training effort.